# 01 · Data

**Owner: Nahid** · Builds `data/daily.csv`, the one file every other notebook reads.

Downloads the S&P 500 index (^GSPC) OHLC, VIX, VIX3M from Yahoo Finance and the 3-month T-bill (DTB3) from FRED, then computes returns, realized variance and the variance risk premium. Conventions are documented at the top of `src/data.py`.

In [ ]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.data import download_raw, build_daily, EPISODES

## Download

In [ ]:
raw = download_raw(start='2004-01-01')   # one extra year so HAR inputs exist by 2005
raw.tail()

## Build the daily dataset

In [ ]:
df = build_daily(raw)
df = df.loc['2005-01-01':]
print(df.shape)
df[['spx_close','vix','vix3m','rf','logret','iv_var','rv_fwd21','vrp']].describe().T

## Checks before anyone uses it

Each check should print `True`. If one fails, fix it before moving on.

In [ ]:
checks = {
    'no missing SPX/VIX': df[['spx_close','vix']].isna().sum().sum() == 0,
    'VIX in percent (10-90 typical)': df['vix'].between(5, 100).all(),
    'VIX3M starts around Dec 2007': df['vix3m'].first_valid_index() < pd.Timestamp('2008-01-31'),
    'rf in decimals': df['rf'].dropna().between(-0.01, 0.10).all(),
    'only the last 21 rows lack rv_fwd21': df['rv_fwd21'].isna().sum() == 21,
    'mean VIX^2 above mean RV (premium exists)': df['iv_var'].mean() > df['rv_fwd21'].mean(),
}
for k, v in checks.items(): print(f'{v!s:5}  {k}')

## Quick look

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, 100*np.sqrt(df['iv_var']), label='VIX (implied vol)', lw=0.8)
ax.plot(df.index, 100*np.sqrt(df['rv_fwd21']), label='Realized vol, next 21 days', lw=0.8)
for name, (s, e) in EPISODES.items():
    ax.axvspan(pd.Timestamp(s), pd.Timestamp(e), color='grey', alpha=0.15)
ax.set_ylabel('annualized vol (%)'); ax.legend(); ax.set_title('Implied vs realized volatility, S&P 500')
plt.show()

## Save

In [ ]:
df.to_csv('../data/daily.csv')
print('saved', len(df), 'rows')